# ML Ops project
*Détecter les caractéristiques d'un visage*

DEFIVES François

GRANIER Sylvain

HACALA Maude

### 0. Vérification des prérequis

In [ ]:
import sys
import torch
import io
import time
import PIL
from PIL import Image
import numpy as np
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import BinaryType


### Import des données annotés 1

Chaque image annottée dans le dossier /data/lot1 est composée d'un fichier csv associé contenant ses caractéristiques.

### Description des données

### Traitement des images

Nous souhaitons redimensionner toutes les images à une taille de 64*64 et segmenter les visages et centrer sur le visage.

In [6]:
# Segmentation simple + redimensionnement des images du dossier
time_start=time.time()
input_folder = 'data/lot1/lot1/'
output_folder = 'data/lot1_resized/'
target_size = (64, 64)

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith('.png'):
        continue

    img_path = os.path.join(input_folder, filename)
    img = Image.open(img_path).convert("RGB")

    # Passage en niveaux de gris pour simplifier
    gray = img.convert("L")
    arr = np.array(gray)

    # Masque : pixels non blancs (ou pas trop blancs)
    # Tu peux ajuster le seuil 250 (0 = noir, 255 = blanc)
    mask = arr < 250

    # Si tout est blanc (image vide), on évite de planter
    if not mask.any():
        cropped = img
    else:
        # Coordonnées des pixels “utiles”
        ys, xs = np.where(mask)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        # Recadrage sur la zone utile
        cropped = img.crop((x_min, y_min, x_max + 1, y_max + 1))

    # Redimensionnement
    img_resized = cropped.resize(target_size)
    img_resized.save(os.path.join(output_folder, filename))
time_end=time.time()
print(f"Temps de traitement des images : {time_end - time_start} secondes")

Temps de traitement des images : 299.8125717639923 secondes


### Spark

In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import BinaryType
import io
# Initialisation de Spark
spark = SparkSession.builder.appName("ImageSegmentation").master("local[*]").getOrCreate()

input_folder = "data/lot1/lot1/"
output_folder = "data/lot1_resized/"
target_size = (64, 64)

os.makedirs(output_folder, exist_ok=True)

# -------------------------------
# UDF : segmentation + crop + resize
# -------------------------------

def segment_and_resize(image_binary):
    img = Image.open(io.BytesIO(image_binary)).convert("RGB")

    gray = img.convert("L")
    arr = np.array(gray)

    # Masque des pixels non-blancs
    mask = arr < 250

    if mask.any():
        ys, xs = np.where(mask)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        cropped = img.crop((x_min, y_min, x_max + 1, y_max + 1))
    else:
        cropped = img  # Image vide → pas de crop

    resized = cropped.resize(target_size)

    # Conversion en binaire pour la sauvegarde
    output = io.BytesIO()
    resized.save(output, format="PNG")
    return output.getvalue()


segment_udf = udf(segment_and_resize, BinaryType())

# -------------------------------
# Lecture des images avec Spark
# -------------------------------
df = spark.read.format("image").load(input_folder)

# df.schema :
# root
#  |-- image: struct
#  |    |-- data: binary
#  |    |-- height: int
#  |    |-- width: int
#  |    |-- nChannels: int
#  |    |-- mode: string
#  |    |-- origin: string

# Application du traitement
processed_df = df.withColumn("processed", segment_udf(col("image.data")))

# -------------------------------
# Sauvegarde
# -------------------------------
rows = processed_df.select("image.origin", "processed").collect()

for row in rows:
    filename = os.path.basename(row["origin"])
    out_path = os.path.join(output_folder, filename)
    with open(out_path, "wb") as f:
        f.write(row["processed"])

print("Terminé !")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Diagnostic Spark & environnement Java/Python
import os, sys, platform, importlib.util
print('JAVA_HOME (env):', os.environ.get('JAVA_HOME'))
print('sys.executable:', sys.executable)
print('Python:', platform.python_version(), platform.architecture())
print('pyspark spec:', importlib.util.find_spec('pyspark'))
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName('diag').master('local[*]').getOrCreate()
    print('Spark OK:', spark)
    spark.stop()
except Exception as e:
    print('Spark erreur:', type(e).__name__, e)
